# Constant

In [1]:
RANDOM_STATE = 74

# Ajustement devoir 1

Liste des changements effectués sur le premier devoir :
- Enlever revenu des attributs cat et du one-hot encoding.
- Enlever la partie qui applique le pipeline sur les données pour le faire ici.
- Mise en place de la division en jeu d'entraînement et jeu de test ici.
- Preparation des pipelines pour les scenarios d'entrainements.

In [2]:
%%capture
%run devoir1.ipynb

# Sans enrichissement
pipeline_noenrich = Pipeline([
    ('preprocessor', ColumnTransformer([
        ("num", numeric_transformer, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ]))
])

# Population seulement
pipeline_population = Pipeline([
    ('enrich', FunctionTransformer(enrichissement_population_pib, validate=False, kw_args={"PIB": False})),
    ('preprocessor', ColumnTransformer([
        ("num", numeric_transformer, num_attribs + ['population']),
        ("cat", OneHotEncoder(), cat_attribs),
    ]))
])

# GDP et Population
pipeline_gdppop = Pipeline([
    ('enrich', FunctionTransformer(enrichissement_population_pib, validate=False, kw_args={"PIB": True})),
    ('preprocessor', ColumnTransformer([
        ("num", numeric_transformer, num_attribs + ['population', 'GDP_inhab']),
        ("cat", OneHotEncoder(), cat_attribs),
    ]))
])

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

customer = pd.read_csv('data/Customer.csv')

# 1. Remove empty labels
customer['revenue'] = customer['revenue'].replace('unknown', np.nan)
customer = customer.dropna(subset=['revenue'])

# 2. Setup labels and remove from subjects
X = customer.drop('revenue', axis=1)
labels = customer['revenue'].copy()
labels = labels.astype(int)

# 3. Setup train and test suits
train_set, test_set, train_label, test_label = train_test_split(X, labels, test_size=0.2, random_state=RANDOM_STATE)

# Introduction
Nom : Potvin
Prénom : Ludovic  

Le but de ce travail est d'entrainer un classeur binaire et un classeur multi-classes. Ensuite, comparer les resultats des deux classeurs

# 2.1 Classeur binaire
Mise en place d'un classeur binaire pour detecter si le sujet est suceptible ou non de produire plus de revenue que la moyenne

## 2.1.1 Function de remplacement revenue
Cette fonction va etre utiliser pour remplacer les label de l'entrainement pour:
- 1: Le revenue est au dessus de la moyenne
- 0: Le revenue est en dessous de la moyenne

In [4]:
labels_mean = labels.mean()

def remplacement_revenue_vers_moyenne(labels):
    labels_bin = (labels > labels_mean).astype(int)

    return labels_bin

## 2.1.2 Echantillonage aleatoire
Cette section vise a creer des echantillons aleatoire de taille 2000, 4000 et 8000.
Les echantillons seront aussi creer pour chaque scenarios.

In [5]:
from sklearn.utils import resample

sample_size = [2000, 4000, 8000]

samples = {
    'noenrich': {},
    'population': {},
    'gdppop': {}
}

labels = {}

train_label_bin = remplacement_revenue_vers_moyenne(train_label)
test_label_bin = remplacement_revenue_vers_moyenne(test_label)

for size in sample_size:
    set_sample, label_sample = resample(
        train_set,
        train_label_bin,
        n_samples=size,
        random_state=RANDOM_STATE,
        stratify=train_label_bin
    )
    
    samples['noenrich'][size] = pipeline_noenrich.fit_transform(set_sample)
    samples['population'][size] = pipeline_population.fit_transform(set_sample)
    samples['gdppop'][size] = pipeline_gdppop.fit_transform(set_sample)

    labels[size] = label_sample

# 2.1.3 Entrainement par validation croise
Ici on va tester les models par validation croise
> TODO document

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate

DTC = DecisionTreeClassifier(random_state=RANDOM_STATE)

results = {
    'noenrich': {},
    'population': {},
    'gdppop': {}
}

for scenario in samples:
    for size in samples[scenario]:
        train_set = samples[scenario][size]
        train_label = labels[size]

        score_k3 = cross_validate(DTC, train_set, train_label, cv=3, scoring=['precision', 'recall', 'f1'])
        score_k10 = cross_validate(DTC, train_set, train_label, cv=10, scoring=['precision', 'recall', 'f1'])
    
        results[scenario][size] = {}
        results[scenario][size]['k3'] = score_k3
        results[scenario][size]['k10'] = score_k10

# 2.1.4 Mesurer les metriques de precision
Affichage des metriques de precision pour chaque scenario et size

In [7]:
scenario_pretty_name = {
    'noenrich': 'non enrichie',
    'population': 'avec population',
    'gdppop': 'avec population et gdp'
}

def calculate_mean(metric):
    return f"{metric.mean():.5f}"

def print_result(label, result):
    precision = calculate_mean(result['test_precision'])
    recall = calculate_mean(result['test_recall'])
    f1 = calculate_mean(result['test_f1'])

    print(f'{label}: {precision}   | {recall} | {f1}')

for scenario in results:
    # Header
    print(f'== Resultat scenario {scenario_pretty_name[scenario]} ==')
    print(f'Name   : Precision | Recall  | F1')
    # K3
    for size in results[scenario]:
        result_k3 = results[scenario][size]['k3']
        print_result(f'k3 {size}', result_k3)

    print()
    # K10
    for size in results[scenario]:
        result_k10 = results[scenario][size]['k10']
        print_result(f'k10 {size}', result_k10)


== Resultat scenario non enrichie ==
Name   : Precision | Recall  | F1
k3 2000: 0.76095   | 0.74578 | 0.75293
k3 4000: 0.80641   | 0.80026 | 0.80307
k3 8000: 0.87567   | 0.87354 | 0.87444

k10 2000: 0.77584   | 0.77567 | 0.77525
k10 4000: 0.81705   | 0.82749 | 0.82134
k10 8000: 0.88539   | 0.88522 | 0.88512
== Resultat scenario avec population ==
Name   : Precision | Recall  | F1
k3 2000: 0.75710   | 0.76654 | 0.76118
k3 4000: 0.81122   | 0.80480 | 0.80781
k3 8000: 0.86778   | 0.86252 | 0.86480

k10 2000: 0.77356   | 0.77940 | 0.77580
k10 4000: 0.82950   | 0.81584 | 0.82196
k10 8000: 0.89235   | 0.88586 | 0.88893
== Resultat scenario avec population et gdp ==
Name   : Precision | Recall  | F1
k3 2000: 0.80140   | 0.78729 | 0.79401
k3 4000: 0.81774   | 0.80999 | 0.81373
k3 8000: 0.87971   | 0.86641 | 0.87292

k10 2000: 0.80563   | 0.79505 | 0.79946
k10 4000: 0.84292   | 0.83980 | 0.84068
k10 8000: 0.89615   | 0.89202 | 0.89397


Avec ces metriques, il est possible de conclure que la meilleur metrique de K est 10.
Il est aussi possible de constater que le scenario avec popualation et gdp performe legerement mieux que les autres scenarios.

# 2.1.5 Calcule d'hyperparametre

In [8]:
from sklearn.model_selection import GridSearchCV

best_model = {
    'noenrich': {},
    'population': {},
    'gdppop': {}
}

param_grid = {
    'max_depth': [3, 5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20, 50]
}

for scenario in samples:
    for size in samples[scenario]:
        train_set = samples[scenario][size]
        train_label = labels[size]

        grid_search = GridSearchCV(
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            param_grid,
            cv=10,
            scoring='f1',
            return_train_score=True
        )

        grid_search.fit(train_set, train_label)

        best_model[scenario][size] = grid_search.best_estimator_

        # Print
        scenario_pn = scenario_pretty_name[scenario]
        score = f"{grid_search.best_score_:.5f}"
        print(f"Meilleur param pour {scenario_pn} - {size}: {grid_search.best_params_} | Score: {score}")

Meilleur param pour non enrichie - 2000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.77525
Meilleur param pour non enrichie - 4000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.82134
Meilleur param pour non enrichie - 8000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.88512
Meilleur param pour avec population - 2000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.77580
Meilleur param pour avec population - 4000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.82196
Meilleur param pour avec population - 8000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.88893
Meilleur param pour avec population et gdp - 2000: {'max_depth': 15, 'min_samples_split': 50} | Score: 0.81392
Meilleur param pour avec population et gdp - 4000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.84068
Meilleur param pour avec population et gdp - 8000: {'max_depth': None, 'min_samples_split': 2} | Score: 0.89397


Avec les resultats precedent, on peut conclure que le meilleur resultat f1 sera avec une max_depth de none et un min sample split de 2.
Ce seront donc les hyperparametres utilise, le modele choisis sera donc le meilleurs model avec population et gdp et un sample size de 8000.

# 2.1.6 Tester le modele
Test le meilleurs modele de l'etape precendente (avec population et gdp et 8000 sujet)
Impression de ses performances

In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score

model = best_model['gdppop'][8000]
X_test = pipeline_gdppop.transform(test_set)
y_test = test_label_bin

y_pred = model.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test Set Results:")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")
print(f"F1-Score: {f1:.5f}")


Test Set Results:
Precision: 0.73241
Recall: 0.71887
F1-Score: 0.72558


# 2.1.7 